# for every item_id,store_id pair
1.sort with date asc

2. perform shift(1) on sales
save

In [26]:

import pandas as pd
import sys
sys.path.append('..')
import importlib
import src.features.lag_features as lf

importlib.reload(lf)

<module 'src.features.lag_features' from 'c:\\Users\\twofi\\Desktop\\inventory-ai-copilot\\notebooks\\..\\src\\features\\lag_features.py'>

In [18]:
#load datasets
sales_df = pd.read_parquet('../data/processed/04_time_features.parquet')

In [19]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 26 columns):
 #   Column          Dtype         
---  ------          -----         
 0   id              str           
 1   item_id         str           
 2   dept_id         str           
 3   cat_id          str           
 4   store_id        str           
 5   state_id        str           
 6   day             str           
 7   sales           uint16        
 8   date            datetime64[us]
 9   wm_yr_wk        uint16        
 10  weekday         str           
 11  wday            uint16        
 12  month           uint16        
 13  year            uint16        
 14  d               str           
 15  event_name_1    str           
 16  event_type_1    str           
 17  event_name_2    str           
 18  event_type_2    str           
 19  snap_CA         uint16        
 20  snap_TX         uint16        
 21  snap_WI         uint16        
 22  sell_price      float64    

In [20]:
cols = ['is_weekend', 'is_month_end', 'is_month_start']

sales_df[cols] = sales_df[cols].astype('int8')

Adding lag features to know yesterdays'(lag_1), a week ago(lag_7), 4week ago(lag_28) sales since all these time interval could be reoccuring or have similar demands


In [25]:
sales_df = lf.add_lag_features(sales_df)

sales_df[['day', 'date', 'sales', 'lag_1', 'lag_7', 'lag_28']].tail(10)

,day,date,sales,lag_1,lag_7,lag_28
58327360,d_1913,2016-04-24,1,3.0,2.0,0.0
58327361,d_1913,2016-04-24,2,0.0,0.0,1.0
58327362,d_1913,2016-04-24,1,0.0,0.0,2.0
58327363,d_1913,2016-04-24,0,4.0,1.0,1.0
58327364,d_1913,2016-04-24,1,2.0,2.0,5.0
58327365,d_1913,2016-04-24,1,0.0,0.0,0.0
58327366,d_1913,2016-04-24,0,1.0,0.0,0.0
58327367,d_1913,2016-04-24,0,1.0,0.0,1.0
58327368,d_1913,2016-04-24,3,1.0,1.0,4.0
58327369,d_1913,2016-04-24,0,0.0,0.0,5.0


In [27]:
sales_df = lf.rolling_mean(sales_df)    

In [28]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 31 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               str           
 1   item_id          str           
 2   dept_id          str           
 3   cat_id           str           
 4   store_id         str           
 5   state_id         str           
 6   day              str           
 7   sales            uint16        
 8   date             datetime64[us]
 9   wm_yr_wk         uint16        
 10  weekday          str           
 11  wday             uint16        
 12  month            uint16        
 13  year             uint16        
 14  d                str           
 15  event_name_1     str           
 16  event_type_1     str           
 17  event_name_2     str           
 18  event_type_2     str           
 19  snap_CA          uint16        
 20  snap_TX          uint16        
 21  snap_WI          uint16        
 22  sel

For `lag_1`, `lag_7`, `lag_28`:

- If they contain missing values at the start of each series, pandas will use float dtype because `NaN` cannot be stored in regular integer columns.
- If you want to keep the `NaN`s, `float32` is usually better than `float64` for memory savings and is accurate enough for lagged sales counts.
- If you can fill or drop the missing values and the values are all integers, a nullable integer dtype like `Int16`/`Int32` or a plain integer dtype is more appropriate.

For rolling mean columns:

- These are averages, so float dtype is required.
- `float32` is generally sufficient unless you need very high precision.

So:
- `lag_*`: float is okay if missing values remain, but prefer `float32` over `float64` for memory; or use nullable ints if you want integer semantics.
- `rolling_mean_*`: float is correct, and `float32` is usually better than `float64` on a large dataset.

In [29]:
cols = ['lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28']
sales_df[cols] = sales_df[cols].astype('float32')

In [30]:
sales_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 31 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               str           
 1   item_id          str           
 2   dept_id          str           
 3   cat_id           str           
 4   store_id         str           
 5   state_id         str           
 6   day              str           
 7   sales            uint16        
 8   date             datetime64[us]
 9   wm_yr_wk         uint16        
 10  weekday          str           
 11  wday             uint16        
 12  month            uint16        
 13  year             uint16        
 14  d                str           
 15  event_name_1     str           
 16  event_type_1     str           
 17  event_name_2     str           
 18  event_type_2     str           
 19  snap_CA          uint16        
 20  snap_TX          uint16        
 21  snap_WI          uint16        
 22  sel

In [31]:
sales_df.to_parquet('../data/processed/05_lag_features.parquet', index=False)